# 01 -- Load a model, set bounds, FBA, FVA

This notebook walks through the sequence from the lecture:

1. load a model and look inside it,
2. set bounds from real numbers,
3. run flux balance analysis (FBA),
4. ask which fluxes are actually determined (flux variability analysis, FVA),

and then asks you to **repeat it on a model of your choice**. There are no fixed
answers -- the point is to get comfortable with the tools.

**Notation used throughout.** Fluxes are in mmol gDW^-1 h^-1, growth rate in h^-1.
Exchange reactions use BiGG IDs such as `EX_glc__D_e` (D-glucose, extracellular).
**Uptake is negative**: a glucose exchange flux of -10 means the cell takes up
glucose. Yields are written Y_p/s (product p per substrate s).

## Part 0 -- Setup

In [ ]:
# Setup -- run this cell first. Safe to re-run: it only installs what is missing.
# If Colab asks you to restart the session after installing, do so and run it again.
import importlib.util
import sys

IN_COLAB = "google.colab" in sys.modules

if importlib.util.find_spec("cobra") is None:
    %pip install -q cobra==0.32.1
if importlib.util.find_spec("escher") is None:
    %pip install -q escher==1.8.1  # optional flux maps; cobra works without it

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()  # lets Colab draw third-party widgets

import cobra
from cobra.io import load_model

_mu = load_model("textbook").slim_optimize()
assert 0.8 < _mu < 0.95, f"Smoke test failed: growth rate {_mu}"
print(f"cobra {cobra.__version__} | Colab: {IN_COLAB} | smoke test {_mu:.4f} h^-1")
print("Setup OK")

`textbook` is the *E. coli* core model (also known as `e_coli_core` in BiGG): central
carbon metabolism plus a lumped biomass reaction. It ships with cobrapy, so it loads
without internet.

In [ ]:
model = load_model("textbook")

## Part 1 -- Look inside

A model is a list of reactions, the metabolites they connect, the genes that encode
them, an objective, and bounds. Calling `model` alone will give you an overview.

In [ ]:
model

The objective is the **biomass reaction**: a single pseudo-reaction that drains
amino acids, nucleotides, lipids and energy in the proportions that make up one gDW
of cells. Its flux is the growth rate in h^-1. Look at what it consumes:

In [ ]:
biomass = model.reactions.get_by_id("Biomass_Ecoli_core")
print(biomass.name)
print(biomass.build_reaction_string(use_metabolite_names=True))

**Exchange reactions** are how the model interacts with the external environment. Their bounds
*are* the growth medium: a negative lower bound means the compound may be taken up. Medium the model operates in is defined by selectively opening the bounds of the exchange reactions.

In [ ]:
import pandas as pd

exchanges = pd.DataFrame(
    [(r.id, r.name, r.lower_bound, r.upper_bound) for r in model.exchanges],
    columns=["id", "name", "lower_bound", "upper_bound"],
).set_index("id")
exchanges

You can check the details of the medium applied to the model as follows:

In [ ]:
model.medium

Note that the medium dictionary above has positive values for the uptake rates, but the model uses negative lower bounds to allow uptake. The dictionary is just a convenient way to specify the medium. For instance, you could set the model to use a different carbon source, acetate, as follows:

In [ ]:
medium = model.medium
medium["EX_glc__D_e"] = 0.0  # remove glucose
medium["EX_ac_e"] = 10.0  # add acetate
model.medium = medium

model.medium

You can achieve the same by directly setting the bounds of the exchange reactions (this time in reverse):

In [ ]:
model.reactions.EX_ac_e.bounds = (0, 1000)
model.reactions.EX_glc__D_e.bounds = (-10, 1000)

model.medium

**Gene-protein-reaction (GPR) rules** link reactions to genes. `or` means
isozymes: either gene is enough. `and` means a complex: every subunit is needed.

In [ ]:
for rxn_id in ["PFK", "PDH"]:
    rxn = model.reactions.get_by_id(rxn_id)
    print(f"{rxn.id:5s} {rxn.name}")
    print(f"      {rxn.reaction}")
    print(f"      GPR: {rxn.gene_reaction_rule}")

## Part 2 -- Set bounds from real numbers

In all likelihood, you will have experiments that give uptake rates of substrates and secretion rates of products to some extent. You can use them (with some conversion) to set the bounds of the exchange reactions to reflect the experimental data on the model. 

For instance, if you measured a glucose uptake rate q_s in mmol gDW^-1 h^-1, you set the lower bound of the glucose exchange reaction to -q_s. The value below is an example -- replace it with your own:

In [ ]:
q_s = 11  # mmol gDW^-1 h^-1 -- example measured glucose uptake rate

model.reactions.EX_glc__D_e.bounds = (-q_s, 1000)  # uptake is NEGATIVE
print(f"EX_glc__D_e bounds: {model.reactions.EX_glc__D_e.bounds}")

Another critical number that matters a lot is aerobicity. If your culture is less than well aerated, limit oxygen uptake.

- **oxygen uptake** (`EX_o2_e`). A very large negative lower bound means "as much as
  the cell wants" -- a well-aerated shake flask, at best. Keep in mind that oxygen constraints are often used to parametrize the models to match experimental data. Leaving it completely open (eg. lower bound -1000) is often unrealistic, and can lead to overestimation of growth rates.


In [ ]:
model.reactions.EX_o2_e.bounds = (-20, 1000)  # example: well aerated, but not unlimited
print(f"EX_o2_e bounds: {model.reactions.EX_o2_e.bounds}")

## Part 3 -- Flux balance analysis

`optimize()` finds a flux distribution that satisfies steady state (S v = 0) and all
bounds, and maximizes the objective.

In [ ]:
solution = model.optimize()
mu = solution.objective_value

print(f"status: {solution.status}")
print(f"growth rate mu = {mu:.4f} h^-1")

`summary()` shows what goes in and out of the cell at this solution. Remember the
sign convention: uptakes are negative.

In [ ]:
model.summary()

Any single flux can be read from `solution.fluxes`. For example, the glycolytic
step phosphofructokinase:

In [ ]:
print(f"PFK flux = {solution.fluxes['PFK']:.3f} mmol gDW^-1 h^-1")

**A yield.** The biomass yield on glucose is growth rate over uptake rate,
Y_x/s = mu / q_s. With mu in h^-1 and q_s in mmol gDW^-1 h^-1 this comes out in
gDW per mmol glucose; the molar mass turns it into gDW per g glucose.

In [ ]:
v_glc = -solution.fluxes["EX_glc__D_e"]  # positive uptake rate
M_glc = model.metabolites.glc__D_e.formula_weight  # g/mol, from the model's formula
Y_xs_mmol = mu / v_glc  # gDW / mmol glucose
Y_xs_g = Y_xs_mmol * 1000 / M_glc  # gDW / g glucose

print(f"Y_x/s = {Y_xs_mmol:.4f} gDW/mmol = {Y_xs_g:.3f} gDW/g glucose")

In [ ]:
# Optional: draw the flux distribution on a map of central metabolism.
# If no map appears (or you see an error), skip it -- nothing else depends on it.
import html
import warnings

try:
    import escher
    from IPython.display import HTML, display

    builder = escher.Builder(
        map_name="e_coli_core.Core metabolism",
        model=model,  # embedded, so the Edit menu can add reactions from it
        reaction_data=solution.fluxes.to_dict(),
        menu="all",
        enable_editing=True,
    )
    # Escher's interactive widget does not load in Colab. Instead, save the map as
    # a standalone web page (flux_map.html, next to this notebook) and show it.
    builder.save_html("flux_map.html")
    with open("flux_map.html") as f:
        page = html.escape(f.read())
    iframe = (
        f'<iframe srcdoc="{page}" width="100%" height="600" '
        'style="border: none"></iframe>'
    )
    with warnings.catch_warnings():
        # IPython suggests IFrame, which needs a served URL that Colab cannot give
        warnings.simplefilter("ignore", UserWarning)
        display(HTML(iframe))
except Exception as err:
    print(f"Escher not available here ({type(err).__name__}: {err})")

## Part 4 -- Is that flux actually determined?

FBA returns **one** optimal flux distribution. Often many distributions reach the
same optimum, and FBA just reports whichever one the solver landed on. **Flux
variability analysis (FVA)** answers: holding growth at its optimum, what is the
smallest and largest value each flux can take?

Start with the by-products: acetate, ethanol, formate, lactate, succinate.

In [ ]:
from cobra.flux_analysis import flux_variability_analysis

byproducts = ["EX_ac_e", "EX_etoh_e", "EX_for_e", "EX_lac__D_e", "EX_succ_e"]

fva_aerobic = flux_variability_analysis(model, byproducts, fraction_of_optimum=1.0)
fva_aerobic["FBA reported"] = solution.fluxes[byproducts]
fva_aerobic.round(4) + 0.0  # "+ 0.0" just tidies -0.0 into 0.0

Where minimum equals maximum, the flux is **determined**: every optimal solution
has that value. Now limit oxygen, as in a dense culture where aeration cannot keep
up, and ask again. `with model:` makes the change temporary -- the bound is restored
when the block ends.

In [ ]:
q_o2_max = 10  # mmol gDW^-1 h^-1 -- example oxygen transfer limit

with model:
    model.reactions.EX_o2_e.lower_bound = -q_o2_max
    sol_lim = model.optimize()
    fva_limited = flux_variability_analysis(model, byproducts, fraction_of_optimum=1.0)
    fva_limited["FBA reported"] = sol_lim.fluxes[byproducts]

print(f"O2-limited growth rate mu = {sol_lim.objective_value:.4f} h^-1")
fva_limited.round(4) + 0.0

In [ ]:
ac = fva_limited.loc["EX_ac_e"]
print(
    f"At mu = {sol_lim.objective_value:.4f} h^-1, acetate secretion can be anywhere "
    f"from {ac.minimum:.2f} to {ac.maximum:.2f} mmol gDW^-1 h^-1. "
    f"FBA happened to report {ac['FBA reported']:.2f}."
)

So whether a flux is determined depends on the constraints, not just on the model.
A single FBA value for a by-product or an internal flux means little until FVA tells
you its range.

The same question for **every** reaction: how many are pinned at the optimum, and
how many are free to vary?

In [ ]:
with model:
    model.reactions.EX_o2_e.lower_bound = -q_o2_max
    fva_all = flux_variability_analysis(model, fraction_of_optimum=1.0)

width = fva_all.maximum - fva_all.minimum
print(f"determined (range < 1e-6): {(width < 1e-6).sum()} of {len(width)} reactions")
fva_all[width >= 1e-6].round(3) + 0.0

Look for ranges that run all the way to a reaction's bound. Those are usually not
real flexibility but a **loop**: reactions that form a cycle with no net effect, so
the solver can push any amount around it without changing growth.

**Parsimonious FBA (pFBA)** adds a second step to FBA: among all solutions with
optimal growth, pick the one with the **smallest total flux**. Loops cost flux and
gain nothing, so pFBA drops them, and the result is a common choice when you need to
report one flux distribution. Note that this is one more assumption (the cell avoids
spending enzyme on flux it does not need), not proof that the other optima are wrong.

In [ ]:
from cobra.flux_analysis import pfba

at_bound = [
    r.id
    for r in model.reactions
    if fva_all.loc[r.id, "maximum"] >= r.upper_bound - 1e-6 and width[r.id] > 1e-6
]
print(f"ranges reaching the upper bound: {at_bound}")

with model:
    model.reactions.EX_o2_e.lower_bound = -q_o2_max
    sol_pfba = pfba(model)

# pfba's objective_value is the total flux, so read growth from the biomass flux
print(f"pFBA growth rate: {sol_pfba.fluxes['Biomass_Ecoli_core']:.4f} h^-1")

# FVA says what is possible at optimal growth; pFBA picks one point inside each range
compare = fva_all.loc[at_bound + byproducts[:3]].copy()
compare["pFBA"] = sol_pfba.fluxes[compare.index]
compare.round(4) + 0.0

`fraction_of_optimum` relaxes the growth requirement: the cell only has to reach
that fraction of its maximum growth rate. Watch the ranges widen.

In [ ]:
with model:
    model.reactions.EX_o2_e.lower_bound = -q_o2_max
    fva_95 = flux_variability_analysis(model, byproducts, fraction_of_optimum=0.95)
fva_95.round(4) + 0.0

## Your turn -- do it again on another model

Repeat Parts 1-4 on a different model. Pick **one** source in the cell below:

- `"bundled"`: `iJO1366`, a genome-scale *E. coli* model that ships with cobrapy.
- `"bigg"`: any model from [BiGG Models](https://bigg.ucsd.edu/models) by its ID,
  e.g. `iML1515` (*E. coli*), `iMM904` (*S. cerevisiae*), `iYO844` (*B. subtilis*).
- `"file"`: your own SBML file (`.xml` or `.xml.gz`). In Colab you will be asked to
  upload it; locally, put it next to this notebook and set `LOCAL_FILE`.

(`load_model("<BiGG ID>")` does not work in this cobrapy version, so the cell
downloads the file itself, and reuses it next time.)

In [ ]:
SOURCE = "bundled"  # "bundled", "bigg" or "file"
MODEL_ID = "iJO1366"  # used for "bundled" and "bigg"
LOCAL_FILE = "my_model.xml"  # for "file" outside Colab: next to this notebook

import gzip
import io
from pathlib import Path

import httpx
from cobra.io import read_sbml_model


def read_model_file(path):
    """Read an SBML file, compressed (.gz) or not."""
    if path.suffix == ".gz":
        return read_sbml_model(io.StringIO(gzip.decompress(path.read_bytes()).decode()))
    return read_sbml_model(str(path))


def load_bigg(model_id):
    path = Path(f"{model_id}.xml.gz")
    if not path.exists():  # download once, then reuse the local copy
        url = f"https://bigg.ucsd.edu/static/models/{model_id}.xml.gz"
        r = httpx.get(url, timeout=180, follow_redirects=True)
        r.raise_for_status()
        path.write_bytes(r.content)
    return read_model_file(path)


if SOURCE == "bundled":
    my_model = load_model(MODEL_ID)
elif SOURCE == "bigg":
    my_model = load_bigg(MODEL_ID)
elif SOURCE == "file":
    if IN_COLAB:
        from google.colab import files

        LOCAL_FILE = next(iter(files.upload()))  # saved next to this notebook
    my_model = read_model_file(Path(LOCAL_FILE))
else:
    raise ValueError(f"unknown SOURCE {SOURCE!r}")

print(
    f"{my_model.id}: {len(my_model.reactions)} reactions, "
    f"{len(my_model.metabolites)} metabolites, {len(my_model.genes)} genes"
)
print(f"objective: {my_model.objective.expression}")

Some questions to work through. Reuse code from above -- most of it works by
changing `model` to `my_model`.

1. **Inspect.** What is the biomass reaction called here? Which compounds can the
   model take up in its default medium? (IDs differ between models: try
   `my_model.reactions.query("glucose", "name")` or
   `my_model.metabolites.query("oxygen", "name")`.)
2. **Bounds.** Set the substrate uptake from your own (or a literature) number.
3. **FBA.** What growth rate does it predict, and what is Y_x/s? How does that
   compare with `textbook`, and why might a bigger model differ?
4. **FVA.** Is acetate (or your by-product of interest) determined at the optimum?
   Does that change when you limit oxygen? Genome-scale FVA on *all* reactions can
   take minutes -- pass a list of reaction IDs.
5. **Try something.** Swap glucose for another carbon source, or knock out a gene
   inside `with my_model:` using `my_model.genes.get_by_id("...").knock_out()`, and
   see what changes.

In [ ]:
# your code here